In [2]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")

y_train = pd.read_csv(
    "../data/y_train.csv"
).squeeze()

In [3]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [4]:
from sklearn.ensemble import RandomForestRegressor

from src.evaluate import rmse_cv

In [5]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_rmse = rmse_cv(
    rf_model,
    X_train,
    y_train
).mean()

print(
    f"Random Forest RMSE: {rf_rmse:.5f}"
)

Random Forest RMSE: 0.13392


In [6]:
X_test = pd.read_csv(
    "../data/X_test.csv"
)

test = pd.read_csv(
    "../data/test.csv"
)

print(X_test.shape)
print(test.shape)

(1459, 318)
(1459, 80)


In [7]:
rf_model.fit(
    X_train,
    y_train
)

print("RF 訓練完成")

RF 訓練完成


In [8]:
from sklearn.model_selection import cross_val_predict

# 1. 計算 Random Forest 的 5-Fold OOF 預測值
print("正在計算 Random Forest OOF...")
oof_rf = cross_val_predict(
    rf_model, 
    X_train, 
    y_train, 
    cv=5, 
    n_jobs=-1
)

# 2. 存檔備用
pd.Series(oof_rf, name="SalePrice").to_csv("rf_oof_train.csv", index=False)
print("RF OOF 已儲存為 rf_oof_train.csv")

正在計算 Random Forest OOF...
RF OOF 已儲存為 rf_oof_train.csv


In [9]:
rf_pred_log = rf_model.predict(X_test)

print(rf_pred_log[:5])

[11.73833846 11.96368241 12.05955764 12.09836178 12.18135594]


In [10]:
import numpy as np

rf_pred = np.expm1(rf_pred_log)

print(rf_pred[:5])

[125283.00212234 156948.97590856 172741.55520863 179576.43301993
 195116.24958842]


In [11]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": rf_pred
})

submission.head()

,Id,SalePrice
0,1461,125283.002122
1,1462,156948.975909
2,1463,172741.555209
3,1464,179576.433020
4,1465,195116.249588


In [12]:
submission.to_csv(
    "../submissions/rf_submission.csv",
    index=False
)

print("RF Submission 已建立")

RF Submission 已建立
